# Python bridge and parameterized character N-gram

Read Chapter 5 core, then Primer-PY. Run this notebook from a fresh kernel before the NumPy practice. Inputs are the unchanged Part I fixtures. MG-0 is a project milestone. No trained neural checkpoint is used.

In [1]:
from pathlib import Path
import sys, json, math
candidates = [Path.cwd(), *Path.cwd().parents]
root = next((p for p in candidates if (p / "data/part-i/ngram.json").is_file()
             and (p / "src/config/book.mjs").is_file()), None)
if root is None:
    raise FileNotFoundError("book repository boundary not found")
sys.path.insert(0, str(root / "code/part-ii"))
print("Repository fixtures found")

Repository fixtures found


## Values, collections and boundaries

The duplicate is an additional observation. Count letters first, then include EOS and reset BOS for each string.

In [2]:
fixture = json.loads((root / "data/part-i/ngram.json").read_text(encoding="utf-8"))
texts = fixture["train"]
counts = {"a": 0, "b": 0, "c": 0}
for text in texts:
    for character in text:
        counts[character] += 1
print(texts, counts)
assert counts == {"a": 3, "b": 2, "c": 1}

['ab', 'ab', 'ac'] {'a': 3, 'b': 2, 'c': 1}


In [3]:
a_row = {"a": 0, "b": 0, "c": 0, "EOS": 0}
for text in texts:
    previous = "BOS"
    for target in list(text) + ["EOS"]:
        if previous == "a":
            a_row[target] += 1
        previous = target
print(a_row)
assert list(a_row.values()) == [0, 2, 1, 0]

{'a': 0, 'b': 2, 'c': 1, 'EOS': 0}


## Inspect one deliberate exception

The exception is handled explicitly so Run All can finish. A missing context and a zero-probability candidate are different conditions.

In [4]:
def smoothed_probability(count, row_total, candidates, alpha):
    denominator = row_total + alpha * candidates
    if denominator == 0:
        raise ValueError("empty unsmoothed context")
    return (count + alpha) / denominator
print(smoothed_probability(2, 3, 4, 1))
try:
    smoothed_probability(0, 0, 4, 0)
except ValueError as error:
    print(str(error))

0.42857142857142855
empty unsmoothed context


## Change order and smoothing

The reference fractions predate this implementation. Compare all six settings, the complete bigram rows, and the exact LCG32 draws. The diagnostic ac overlaps training and is not a generalization test.

In [5]:
from ngram import fit, probabilities, sequence_probability, generate, run
results = run()

{
  "held_out": [
    {
      "n": 1,
      "alpha": 0,
      "ac": 0.012345679012345678,
      "aa": 0.037037037037037035
    },
    {
      "n": 2,
      "alpha": 0,
      "ac": 0.3333333333333333,
      "aa": 0.0
    },
    {
      "n": 3,
      "alpha": 0,
      "ac": 0.3333333333333333,
      "aa": 0.0
    },
    {
      "n": 1,
      "alpha": 1,
      "ac": 0.014565316340464271,
      "aa": 0.029130632680928543
    },
    {
      "n": 2,
      "alpha": 1,
      "ac": 0.06530612244897958,
      "aa": 0.011661807580174925
    },
    {
      "n": 3,
      "alpha": 1,
      "ac": 0.06530612244897958,
      "aa": 0.02040816326530612
    }
  ],
  "generation": {
    "tokens": [
      "a",
      "b",
      "EOS"
    ],
    "draws": [
      0.2523451747838408,
      0.08812504541128874,
      0.5772811982315034
    ],
    "stop": "eos"
  }
}


In [6]:
n, alpha = 2, 1
model = fit(texts, fixture["vocabulary"], n=n, alpha=alpha)
print("candidate order:", fixture["vocabulary"])
print("after a:", probabilities(model, ["a"]))
print("P(ab, including EOS):", sequence_probability(model, "ab"))
assert math.isclose(sequence_probability(model, "ab"), 6/49)
assert probabilities(fit(texts, fixture["vocabulary"], 3, 0), ["a", "a"]) is None

candidate order: ['a', 'b', 'c', 'EOS']
after a: [0.14285714285714285, 0.42857142857142855, 0.2857142857142857, 0.14285714285714285]
P(ab, including EOS): 0.12244897959183672


## Transfer exercise and visible reference result

Remove one copy of ab in a separate variant. The a row should be [0,1,1,0] and P(b|a)=1/2 without smoothing.

In [7]:
variant = fit(["ab", "ac"], fixture["vocabulary"], n=2, alpha=0)
print(probabilities(variant, ["a"]))
assert probabilities(variant, ["a"]) == [0, 0.5, 0.5, 0]

[0.0, 0.5, 0.5, 0.0]
